In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

ytrain = pd.read_csv('/kaggle/input/competitions/iaio-2026-sfr-timeseries-classification/ytrain.csv')

def make_feats(df):
    feats = []
    feats.append(df.mean(axis=0).values)
    feats.append(df.median(axis=0).values)
    feats.append(df.mode(axis=0).iloc[0].values)
    feats.append(df.skew(axis=0).values)
    feats.append(df.kurtosis(axis=0).values)
    feats.append([len(df)])
    feats.append(df.isna().sum().values)
    feats.append(df.std(axis=0).values)
    return np.concatenate(feats)

X_train = []
y_train = []
train_dir = Path('/kaggle/input/competitions/iaio-2026-sfr-timeseries-classification/xtrain/xtrain')

for idx, row in tqdm(ytrain.iterrows(), total=len(ytrain)):
    file_path = train_dir / f"{row['Id']}.csv"
    df= pd.read_csv(file_path)
    if file_path.exists():
        X_train.append(make_feats(df))
        y_train.append(row['Attack'])

X_train = np.array(X_train)
y_train = np.array(y_train)

sample_submission = pd.read_csv('/kaggle/input/competitions/iaio-2026-sfr-timeseries-classification/sample_submission.csv')
test_ids = sample_submission['Id'].values
X_test = []
test_dir = Path('/kaggle/input/competitions/iaio-2026-sfr-timeseries-classification/xtest/xtest')

for test_id in tqdm(test_ids):
    file_path = test_dir / f"{test_id}.csv"
    df = pd.read_csv(file_path)
    if file_path.exists():
        X_test.append(make_feats(df))
    else:
        X_test.append(np.zeros(X_train.shape[1]))

X_test = np.array(X_test)

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

In [24]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
X_train= scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
from catboost import CatBoostClassifier
cat = CatBoostClassifier(n_estimators = 250, max_depth =10, random_state = 24,).fit(X_train, y_train)

Learning rate set to 0.010762
0:	learn: 0.6871027	total: 292ms	remaining: 2m 25s
1:	learn: 0.6815961	total: 584ms	remaining: 2m 25s
2:	learn: 0.6760492	total: 877ms	remaining: 2m 25s
3:	learn: 0.6693787	total: 1.17s	remaining: 2m 24s
4:	learn: 0.6632029	total: 1.46s	remaining: 2m 24s
5:	learn: 0.6572465	total: 1.75s	remaining: 2m 24s
6:	learn: 0.6498000	total: 2.05s	remaining: 2m 24s
7:	learn: 0.6428720	total: 2.34s	remaining: 2m 24s
8:	learn: 0.6378942	total: 2.63s	remaining: 2m 23s
9:	learn: 0.6313918	total: 2.92s	remaining: 2m 23s
10:	learn: 0.6244647	total: 3.21s	remaining: 2m 22s
11:	learn: 0.6177858	total: 3.51s	remaining: 2m 22s
12:	learn: 0.6117837	total: 3.8s	remaining: 2m 22s
13:	learn: 0.6055597	total: 4.09s	remaining: 2m 21s
14:	learn: 0.5987265	total: 4.38s	remaining: 2m 21s
15:	learn: 0.5936770	total: 4.68s	remaining: 2m 21s
16:	learn: 0.5871178	total: 4.97s	remaining: 2m 21s
17:	learn: 0.5816389	total: 5.26s	remaining: 2m 20s
18:	learn: 0.5766521	total: 5.56s	remaining: 

In [43]:
predictions = cat.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({'Id': test_ids, 'Attack': predictions})

In [44]:
submission.to_csv("sub4.csv",index=False)